In [1]:
import pandas as pd
from astropy.table import Table
import matplotlib.pyplot as plt
from matplotlib import cm
from matplotlib.colors import Normalize
from matplotlib.colors import TwoSlopeNorm, LinearSegmentedColormap
from astropy.stats import sigma_clip
from clean_lc_paper_1 import *
from basics import *
import numpy as np

In [2]:
ds_bad_times = [131.43378, 181.60556, 230.40734, 255.43297, 256.19620, 258.60637, 280.65946, 281.18166, 290.66168,
                321.59224, 383.57387, 397.03067, 400.72203, 442.41804, 475.19640, 503.95780, 538.38311, 566.70265,
                598.47678, 629.44327, 660.45417, 690.46083, 711.14815, 761.27976, 802.52966, 845.14957, 874.27250,
                886.76523, 887.44811, 906.04662, 937.05752, 969.43418, 1032.4560, 1063.3865, 1070.0949, 1099.2981,
                1116.8522, 1118.6599, 1121.3914, 1122.0341, 1126.0511, 1154.0492, 1160.7576, 1182.1278, 1215.0626,
                1245.0291, 1273.1879, 1305.0424, 1336.4148, 1371.4828, 1404.4980, 1435.5089, 1471.2598, 1525.0870,
                1558.0662, 1559.1508, 1591.2463, 130.43898, 169.63898, 183.55898, 200.31608, 231.19898, 246.18889,
                260.11898, 291.31898, 323.43898, 349.63898, 352.23898, 372.75898, 384.15898, 443.31898, 476.15898,
                504.43898, 539.35898, 567.27898, 599.79898, 630.07898, 660.91898, 691.03898, 719.75898, 735.19898,
                763.59898, 808.39898, 845.55898, 874.63898, 906.71898, 937.71898, 969.99898, 1001.0390, 1032.9590,
                1064.0390, 1072.4390, 1098.4390, 1126.6390, 1154.7990, 1164.8790, 1182.5990, 1215.5990, 1245.7590,
                1274.0790, 1289.7190, 1296.1990, 1306.0390, 1337.1190, 1373.2390, 1405.1990, 1413.0390, 1418.3190,
                1436.1590, 1471.9590, 1477.5990, 1488.4390, 1525.9590, 1581.9190, 1586.6390]
ds_bad_times.sort()

In [ ]:
pd.DataFrame(ds_bad_times).to_csv('../data/ds_bad_times.csv',index=False,header=None)


In [ ]:
def calc_P_extrema_smooth(time, flux):
    time = np.asarray(time)
    flux = np.asarray(flux)

    # Remove NaNs
    valid = np.isfinite(time) & np.isfinite(flux)
    time = time[valid]
    flux = flux[valid]

    flux = smooth_v2(flux)

    # Point-to-point differences
    dflux = np.diff(flux)

    # Iterative 4-sigma clipping
    clipped = sigma_clip(
        dflux,
        sigma=4.0,
        maxiters=None,
        masked=True
    )

    # Sign of each difference; retain masked positions
    signs = np.sign(clipped.data)
    good = ~clipped.mask & (signs != 0) #mask of good data points (i.e. not removed by sigma clipping) 

    # Count crossings only between adjacent unmasked differences
    adjacent_good = good[:-1] & good[1:]
    crossings = (
        adjacent_good &
        (signs[:-1] != signs[1:])
    )

    N_extrema = np.sum(crossings)

    if N_extrema == 0:
        return np.nan

    N = len(flux)
    dt = 29.4/(24 * 60)  # days

    P_extrema = 2 * N * dt / N_extrema

    ###get time of crossings for plots
    # Convert clipped result to arrays
    dflux_values = clipped.data
    dflux_mask = np.ma.getmaskarray(clipped)

    # Signs of the point-to-point differences
    signs = np.sign(dflux_values)

    # Valid adjacent pairs
    valid_pairs = (
        ~dflux_mask[:-1]
        & ~dflux_mask[1:]
        & (signs[:-1] != 0)
        & (signs[1:] != 0)
    )

    # Sign changes = extrema
    crossing_idx = np.where(
        valid_pairs & (signs[:-1] != signs[1:])
    )[0]

    # Times corresponding to dflux values
    dflux_time = time[1:]

    # Interpolate the zero-crossing time
    t1 = dflux_time[crossing_idx]
    t2 = dflux_time[crossing_idx + 1]
    y1 = dflux_values[crossing_idx]
    y2 = dflux_values[crossing_idx + 1]

    crossing_time = t1 - y1 * (t2 - t1) / (y2 - y1)

    # Approximate flux at the extrema
    extrema_flux = np.interp(crossing_time, time, flux)

    return P_extrema, crossing_time, extrema_flux

In [ ]:

def smooth_and_count_crossings(
    time,
    flux,
    bad_times,
    width_days=2
):
    time = np.asarray(time, dtype=float)
    flux = np.asarray(flux, dtype=float)
    bad_times = np.sort(np.asarray(bad_times, dtype=float))

    smoothed = np.full_like(flux, np.nan)
    crossing_times = []
    crossing_counts = []
    segment_ranges = []

    cadence = 29.4 / (60 * 24)
    filter_width = max(1, int(round(width_days / cadence)))
    kernel = Box1DKernel(filter_width)

    # Convert bad timestamps into array indices
    boundary_indices = np.searchsorted(time, bad_times)
    boundary_indices = boundary_indices[
        (boundary_indices > 0) & (boundary_indices < len(time))
    ]

    starts = np.r_[0, boundary_indices]
    ends = np.r_[boundary_indices, len(time)]

    for start, end in zip(starts, ends):
        segment_time = time[start:end]
        segment_flux = flux[start:end]

        valid = np.isfinite(segment_time) & np.isfinite(segment_flux)

        if np.sum(valid) < 3:
            crossing_counts.append(0)
            segment_ranges.append((start, end))
            continue

        t = segment_time[valid]
        f = segment_flux[valid]

        # Smooth this segment only
        f_smooth = convolve(
            f,
            kernel,
            boundary="extend"
        )

        f_smooth = convolve(
            f_smooth,
            kernel,
            boundary="extend"
        )

        smoothed[start:end][valid] = f_smooth

        # Define crossings relative to the segment median
        crossing_level = np.median(f_smooth)
        centred_flux = f_smooth - crossing_level

        # Prevent exact zeros from creating spurious crossings
        signs = np.sign(centred_flux)
        signs[signs == 0] = np.nan

        crossing_mask = (
            np.isfinite(signs[:-1])
            & np.isfinite(signs[1:])
            & (signs[:-1] != signs[1:])
        )

        crossing_t = t[:-1][crossing_mask]
        crossing_times.extend(crossing_t)

        crossing_counts.append(len(crossing_t))
        segment_ranges.append((start, end))

    return (
        smoothed,
        np.asarray(crossing_times),
        np.asarray(crossing_counts),
        segment_ranges
    )

In [ ]:
def smooth_v2(flux, width_days=2):
    flux = np.asarray(flux, dtype=float)
    smoothed = np.full_like(flux, np.nan)

    cadence = 29.4 / (60 * 24)  # days
    points_per_day = 1 / cadence
    filter_width = max(1, int(round(width_days * points_per_day)))

    kernel = Box1DKernel(filter_width)

    finite = np.isfinite(flux)

    # Find contiguous finite regions
    starts = np.flatnonzero(finite & ~np.r_[False, finite[:-1]])
    ends = np.flatnonzero(finite & ~np.r_[finite[1:], False]) + 1

    for start, end in zip(starts, ends):
        segment = flux[start:end]

        smoothed_segment = convolve(
            segment,
            kernel,
            boundary="extend",
            nan_treatment="interpolate"
        )

        # Apply the filter twice, as in your original function
        smoothed_segment = convolve(
            smoothed_segment,
            kernel,
            boundary="extend",
            nan_treatment="interpolate"
        )

        smoothed[start:end] = smoothed_segment

    return smoothed

In [ ]:
def smooth(flux, width_days=2):
    cadence = 29.4/(60 * 24) #cadence in days
    points_per_day = int(1 / cadence)

    filter_width = width_days * points_per_day  # 10 day high pass filter

    highly_smoothed_flux1 = convolve(flux, Box1DKernel(filter_width), boundary='extend')
    smoothed = convolve(highly_smoothed_flux1, Box1DKernel(filter_width), boundary='extend')

    return smoothed

In [ ]:
#zach claytor RGs (have a measured period for these)
zc = pd.read_csv('../data/claytor_rgs.csv')
test = zc[zc['kbonus_designation']=='kic-007904418'] #chosen because it has oscillations and a hint of nastiness. numax = 0.86 muHz

In [ ]:
fname = test['kbonus_designation'].iloc[0]
num_id = fname.split('-')[-1]
P_zc = 1000000/(test['numax'].iloc[0] * 60 * 60 * 24) #x10^6 to convert to Hertz, then take reciprocal to get period in days

In [ ]:
#open fits file and download lightcurve quarters
filepath = f'/Users/z5214708/tess_data/stello/Kepler/LC/KBonus/kbonus-bkg/lcs/{num_id[0:4]}/{num_id}/hlsp_kbonus-bkg_kepler_kepler_{fname}_kepler_v1.0_lc.fits'
stitched = Table.read(filepath, hdu=1, format='fits', unit_parse_strict='silent')

#remove NaNs
nan_mask = np.isnan(stitched['SAP_FLUX'])
time = np.asarray(stitched['TIME'][~nan_mask])
flux = np.asarray(stitched['SAP_FLUX'][~nan_mask])

#split into quarters
time_q, flux_q = split_light_curve(time,flux)

#calculate p extrema per quarter
p_extrema_vals = []
for i in range(len(time_q)):
    p_extrema_vals.append(calc_P_extrema_smooth(time_q[i],flux_q[i])[0])

# Concatenate all quarters into one light curve
time = np.concatenate(time_q)
flux = np.concatenate(flux_q)

p_extrema_tot = calc_P_extrema_smooth(time,flux)[0]

# Normalize the p_extrema values to [0, 1] for colormap mapping
norm = Normalize(vmin=min(p_extrema_vals), vmax=max(p_extrema_vals))
cmap = cm.RdYlBu

#plot lc with each quarter coloured by its P_extrema val
plt.figure(figsize=(12, 8))
for i in range(len(time_q)):
    color = cmap(norm(p_extrema_vals[i]))
    plt.scatter(time_q[i], flux_q[i], s=1, color=color)

# Add a colorbar to show the mapping
sm = cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
cbar = plt.colorbar(sm, ax=plt.gca())
cbar.set_label('P_extrema (days)')

plt.xlabel('Time [Days]')
plt.ylabel('Flux')
plt.title(f'P_ZC = {P_zc:.2f} days. P_extrema_tot = {p_extrema_tot:.2f} days.')
plt.savefig('../figures/calc_p_extrema/7904418_p_ext_smooth.png',bbox_inches='tight')

# Q2

In [ ]:
time = time_q[2]
flux = flux_q[2]

p_extrema, crossing_time, extrema_flux = calc_P_extrema_smooth(time,flux)

smoothed = smooth(flux)

# Point-to-point differences
dflux = np.diff(smoothed)
dflux_time = time[1:]

# Iterative 4-sigma clipping
clipped = sigma_clip(
    dflux,
    sigma=4.0,
    maxiters=None,
    masked=True
)

In [ ]:
fig, axs = plt.subplots(
    3, 1,
    figsize=(12, 12),
    sharex=True,
    gridspec_kw={'hspace': 0.03}
)

# Light curve
axs[0].plot(time, flux, color='black', lw=0.8)
axs[0].scatter(
    crossing_time,
    extrema_flux,
    color='red',
    s=35,
    zorder=5,
    label='Detected extrema'
)
axs[0].legend()

# Light curve
axs[1].plot(time, smoothed, color='black', lw=0.8)
axs[1].scatter(
    crossing_time,
    extrema_flux,
    color='red',
    s=35,
    zorder=5,
    label='Detected extrema'
)


# Clipped differences
axs[2].plot(dflux_time, clipped, color='black', lw=0.8)
axs[2].axhline(0, color='grey', ls='--', lw=0.8)
axs[2].scatter(
    crossing_time,
    np.zeros_like(crossing_time),
    color='red',
    s=35,
    zorder=5
)

axs[0].set_ylabel('Raw Flux', fontsize=16)
axs[1].set_ylabel('Smoothed Flux', fontsize=16)
axs[2].set_ylabel(r'PTP(Flux) ($4\sigma$ clipped)', fontsize=16)
axs[2].set_xlabel('Time [Days]', fontsize=16)

for ax in axs:
    ax.tick_params(labelsize=12)

axs[0].set_title(f'P_extrema = {p_extrema:.2f} Days',fontsize=18)

plt.savefig('../figures/calc_p_extrema/7904418_q2_smoothed_crossings.png',bbox_inches='tight')

In [ ]:
time = time_q[2]
flux = flux_q[2]

p_extrema, crossing_time, extrema_flux = calc_P_extrema_smooth(time,flux)

smoothed = smooth_v2(flux)

# Point-to-point differences
dflux = np.diff(smoothed)
dflux_time = time[1:]

# Iterative 4-sigma clipping
clipped = sigma_clip(
    dflux,
    sigma=4.0,
    maxiters=None,
    masked=True
)

In [ ]:
fig, axs = plt.subplots(
    3, 1,
    figsize=(12, 12),
    sharex=True,
    gridspec_kw={'hspace': 0.03}
)

# Light curve
axs[0].scatter(time, flux, color='black', s=0.05)
axs[0].scatter(
    crossing_time,
    extrema_flux,
    color='red',
    s=35,
    zorder=5,
    label='Detected extrema'
)
axs[0].legend()

# Light curve
axs[1].scatter(time, smoothed, color='black', s=0.05)
axs[1].scatter(
    crossing_time,
    extrema_flux,
    color='red',
    s=35,
    zorder=5,
    label='Detected extrema'
)


# Clipped differences
axs[2].plot(dflux_time, clipped, color='black', lw=0.8)
axs[2].axhline(0, color='grey', ls='--', lw=0.8)
axs[2].scatter(
    crossing_time,
    np.zeros_like(crossing_time),
    color='red',
    s=35,
    zorder=5
)

axs[0].set_ylabel('Raw Flux', fontsize=16)
axs[1].set_ylabel('Smoothed Flux', fontsize=16)
axs[2].set_ylabel(r'PTP(Flux) ($4\sigma$ clipped)', fontsize=16)
axs[2].set_xlabel('Time [Days]', fontsize=16)

for ax in axs:
    ax.tick_params(labelsize=12)

axs[0].set_title(f'P_extrema = {p_extrema:.2f} Days',fontsize=18)

plt.savefig('../figures/calc_p_extrema/7904418_q2_smoothed_v2_crossings.png',bbox_inches='tight')

In [ ]:
time = time_q[2]
flux = flux_q[2]

p_extrema, crossing_time, extrema_flux = calc_P_extrema_smooth(time,flux)

smoothed = smooth_v3(time,flux)

# Point-to-point differences
dflux = np.diff(smoothed)
dflux_time = time[1:]

# Iterative 4-sigma clipping
clipped = sigma_clip(
    dflux,
    sigma=4.0,
    maxiters=None,
    masked=True
)

In [ ]:
fig, axs = plt.subplots(
    3, 1,
    figsize=(12, 12),
    sharex=True,
    gridspec_kw={'hspace': 0.03}
)

# Light curve
axs[0].scatter(time, flux, color='black', s=0.05)
axs[0].scatter(
    crossing_time,
    extrema_flux,
    color='red',
    s=35,
    zorder=5,
    label='Detected extrema'
)
axs[0].legend()

# Light curve
axs[1].scatter(time, smoothed, color='black', s=0.05)
axs[1].scatter(
    crossing_time,
    extrema_flux,
    color='red',
    s=35,
    zorder=5,
    label='Detected extrema'
)


# Clipped differences
axs[2].plot(dflux_time, clipped, color='black', lw=0.8)
axs[2].axhline(0, color='grey', ls='--', lw=0.8)
axs[2].scatter(
    crossing_time,
    np.zeros_like(crossing_time),
    color='red',
    s=35,
    zorder=5
)

axs[0].set_ylabel('Raw Flux', fontsize=16)
axs[1].set_ylabel('Smoothed Flux', fontsize=16)
axs[2].set_ylabel(r'PTP(Flux) ($4\sigma$ clipped)', fontsize=16)
axs[2].set_xlabel('Time [Days]', fontsize=16)

for ax in axs:
    ax.tick_params(labelsize=12)

axs[0].set_title(f'P_extrema = {p_extrema:.2f} Days',fontsize=18)

plt.savefig('../figures/calc_p_extrema/7904418_q2_smoothing_fail.png',bbox_inches='tight')

In [ ]:
import numpy as np
from astropy.convolution import convolve, Box1DKernel
from astropy.stats import sigma_clip


def calc_P_extrema_smooth(
    time,
    flux,
    bad_times,
    width_days=2,
    include_time_gaps=True,
    time_gap_factor=1.5
):
    time = np.asarray(time, dtype=float)
    flux = np.asarray(flux, dtype=float)

    # Preserve the original array length for plotting
    smoothed_flux = np.full_like(flux, np.nan)

    # Remove NaNs for processing
    valid = np.isfinite(time) & np.isfinite(flux)
    original_indices = np.flatnonzero(valid)

    t = time[valid]
    f = flux[valid]

    if len(t) < 3:
        return np.nan, np.array([]), np.array([]), np.array([]), smoothed_flux

    # Kepler long-cadence sampling
    cadence = 29.4 / (24 * 60)
    filter_width = max(1, int(round(width_days / cadence)))
    kernel = Box1DKernel(filter_width)

    # ------------------------------------------------------------
    # Find segment boundaries
    # ------------------------------------------------------------

    boundaries = []

    # Boundaries specified by bad timestamps
    bad_times = np.sort(np.asarray(bad_times, dtype=float))
    bad_indices = np.searchsorted(t, bad_times, side="left")

    boundaries.extend(
        bad_indices[(bad_indices > 0) & (bad_indices < len(t))]
    )

    # Boundaries caused by NaNs in the original light curve
    nan_gap_indices = np.flatnonzero(
        np.diff(original_indices) > 1
    ) + 1

    boundaries.extend(nan_gap_indices)

    # Boundaries caused by time discontinuities
    if include_time_gaps:
        time_gap_indices = np.flatnonzero(
            np.diff(t) > time_gap_factor * cadence
        ) + 1

        boundaries.extend(time_gap_indices)

    boundaries = np.unique(boundaries)

    segment_starts = np.r_[0, boundaries]
    segment_ends = np.r_[boundaries, len(t)]

    crossing_times_all = []
    extrema_flux_all = []
    crossing_counts = []

    total_number_of_points = 0

    # ------------------------------------------------------------
    # Smooth and analyse each segment independently
    # ------------------------------------------------------------

    for start, end in zip(segment_starts, segment_ends):

        segment_time = t[start:end]
        segment_flux = f[start:end]

        total_number_of_points += len(segment_flux)

        if len(segment_flux) < 3:
            crossing_counts.append(0)
            continue

        # Smooth this segment only
        segment_smoothed = convolve(
            segment_flux,
            kernel,
            boundary="extend"
        )

        segment_smoothed = convolve(
            segment_smoothed,
            kernel,
            boundary="extend"
        )

        # Store smoothed result in its original position
        smoothed_flux[original_indices[start:end]] = segment_smoothed

        # Point-to-point differences
        dflux = np.diff(segment_smoothed)

        if len(dflux) < 2:
            crossing_counts.append(0)
            continue

        # Iterative 4-sigma clipping
        clipped = sigma_clip(
            dflux,
            sigma=4.0,
            maxiters=None,
            masked=True
        )

        dflux_values = np.asarray(clipped.data)
        dflux_mask = np.ma.getmaskarray(clipped)

        # Signs of the differences
        signs = np.sign(dflux_values)

        good = (
            ~dflux_mask
            & np.isfinite(dflux_values)
            & (signs != 0)
        )

        # Only count crossings between adjacent good differences
        adjacent_good = good[:-1] & good[1:]

        crossing_idx = np.where(
            adjacent_good
            & (signs[:-1] != signs[1:])
        )[0]

        crossing_counts.append(len(crossing_idx))

        if len(crossing_idx) == 0:
            continue

        # dflux[i] corresponds approximately to time[i + 1]
        dflux_time = segment_time[1:]

        t1 = dflux_time[crossing_idx]
        t2 = dflux_time[crossing_idx + 1]

        y1 = dflux_values[crossing_idx]
        y2 = dflux_values[crossing_idx + 1]

        denominator = y2 - y1
        nonzero = denominator != 0

        t1 = t1[nonzero]
        t2 = t2[nonzero]
        y1 = y1[nonzero]
        y2 = y2[nonzero]

        # Linearly interpolate the zero-crossing time
        crossing_time = t1 - y1 * (t2 - t1) / (y2 - y1)

        # Flux at each estimated extremum
        extrema_flux = np.interp(
            crossing_time,
            segment_time,
            segment_smoothed
        )

        crossing_times_all.extend(crossing_time)
        extrema_flux_all.extend(extrema_flux)

    # Convert results to arrays
    crossing_time = np.asarray(crossing_times_all)
    extrema_flux = np.asarray(extrema_flux_all)

    # Sort crossings chronologically
    sort_order = np.argsort(crossing_time)
    crossing_time = crossing_time[sort_order]
    extrema_flux = extrema_flux[sort_order]

    N_extrema = len(crossing_time)

    if N_extrema == 0:
        P_extrema = np.nan
    else:
        # Same convention as your original calculation
        P_extrema = (
            2 * total_number_of_points * cadence / N_extrema
        )

    return (
        P_extrema,
        crossing_time,
        extrema_flux,
        np.asarray(crossing_counts),
        smoothed_flux
    )

In [ ]:
p_extrema, crossing_time, extrema_flux, crossing_counts, smoothed = (
    calc_P_extrema_smooth(
        time,
        flux,
        bad_times=ds_bad_times
    )
)

In [ ]:
plt.plot(time, flux, alpha=0.4)
plt.plot(time, smoothed, color="black")
plt.scatter(
    crossing_time,
    extrema_flux,
    color="red",
    zorder=3
)